# PyMAUDE — Example Notebook

**PyMAUDE** is a Python library for accessing and analyzing FDA MAUDE (Manufacturer and User Facility Device Experience) adverse event data via a fast [DuckDB](https://duckdb.org/) backend.

This notebook downloads real FDA MAUDE data and demonstrates all major library capabilities using venous stent and rotational thrombectomy devices as worked examples.

> **First run:** `add_years(..., download=True)` downloads zip files from the FDA FTP server (~few hundred MB for 3 years of device/master/text). Subsequent runs use the local cache and skip unchanged files via MD5 checksum.

---
## Contents
1. [Setup](#1-setup)
2. [Download & load data](#2-load)
3. [Inspect the database](#3-inspect)
4. [Exact-field queries](#4-exact)
5. [Substring search](#5-search)
6. [Grouped search across device classes](#6-grouped)
7. [Event narratives](#7-narratives)
8. [Enrich with patient outcomes](#8-patient)
9. [Enrich with device problem codes](#9-problems)
10. [Trend analysis by year](#10-trends)
11. [Raw SQL](#11-sql)
12. [PRISMA-compliant search strategy](#12-prisma)

---
## 1. Setup <a id="1-setup"></a>

In [3]:
from pymaude import MaudeDatabase, DeviceSearchStrategy

DB_PATH   = './maude.duckdb'   # persistent DuckDB file
DATA_DIR  = './maude_data'     # downloaded zip/txt files live here
YEARS     = '2021-2022'        # adjust to taste; more years = more data


# tmp
DATA_DIR  = './maude_data'

---
## 2. Download & load data <a id="2-load"></a>

Pass `download=True` to fetch from the FDA FTP area. Files are cached locally — re-running this cell only downloads files whose MD5 checksum has changed since the last run.

In [4]:
db = MaudeDatabase(DB_PATH, data_dir=DATA_DIR, verbose=True, memory_limit='2GB')

db.add_years(
    YEARS,
    # tables=['master', 'device', 'text', 'patient', 'problems'],
    tables=['master', 'device', 'text', 'problems'],
    download=True
)

  Using cached device2021.zip
  Loading device 2021...
    2,032,832 rows
  Using cached device2022.zip
  Loading device 2022...
    2,954,957 rows
  Loading master (2021–2022) from cumulative file...
    4,973,245 total rows
  Loading problems (all records)...
    23,358,818 rows
  Loading text 2021...
    4,872,211 rows
  Loading text 2022...
    11,512,328 rows


To pull in the latest monthly FDA updates at any point:
```python
db.update(download=True)
```

---
## 3. Inspect the database <a id="3-inspect"></a>

In [5]:
db.info()

Database : ./maude.duckdb
Data dir : ./maude_data
  master    :  4,973,245 rows  (2021–2022)
  device    :  4,987,789 rows  (2021–2022)
  text      : 11,512,328 rows
  patient   : not loaded
  problems  : 23,358,818 rows


In [7]:
db.add_years('2026', download=True)

  Loading device 2026...
    2,078,289 rows
  Loading master (2026) from cumulative file...
    2,077,953 total rows
  Loading patient (all records)...
    2,079,144 rows
  Loading text 2026...
    15,374,117 rows


---
## 4. Exact-field queries <a id="4-exact"></a>

`query_device()` does **exact, case-insensitive** matching. Combine any of `brand_name`, `generic_name`, `manufacturer_name`, `product_code`, `start_date`, `end_date`.

In [9]:
# All events for a specific product code (NIQ = venous stents)
niq = db.query_device(product_code='NIQ')
print(f'NIQ (venous stent) events loaded: {len(niq):,}')
niq[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'MANUFACTURER_D_NAME', 'DATE_RECEIVED']].head(10)

NIQ (venous stent) events loaded: 10,980


,MDR_REPORT_KEY,BRAND_NAME,GENERIC_NAME,MANUFACTURER_D_NAME,DATE_RECEIVED
0,16077954,ONYX TRUCOR,CORONARY DRUG-ELUTING STENT,MEDTRONIC IRELAND,2022-12-30
1,16073415,XIENCE SKYPOINT DRUG ELUTING CORONARY STENT DE...,DRUG ELUTING CORONARY STENT DELIVERY SYSTEM,ABBOTT VASCULAR,2022-12-29
2,16064687,SYNERGY XD,CORONARY DRUG-ELUTING STENT,BOSTON SCIENTIFIC CORPORATION,2022-12-28
3,16064318,PROMUS PREMIER,"STENT, CORONARY, DRUG-ELUTING",BOSTON SCIENTIFIC CORPORATION,2022-12-28
4,16063289,PROMUS PREMIER SELECT,"STENT, CORONARY, DRUG-ELUTING",BOSTON SCIENTIFIC CORPORATION,2022-12-28
5,16061817,PROMUS ELITE,"STENT, CORONARY, DRUG-ELUTING",BOSTON SCIENTIFIC CORPORATION,2022-12-28
6,16057387,SYNERGY,CORONARY DRUG-ELUTING STENT,BOSTON SCIENTIFIC CORPORATION,2022-12-27
7,16054033,SYNERGY,CORONARY DRUG-ELUTING STENT,BOSTON SCIENTIFIC CORPORATION,2022-12-27
8,16054010,SYNERGY,CORONARY DRUG-ELUTING STENT,BOSTON SCIENTIFIC CORPORATION,2022-12-27
9,16052877,PROMUS PREMIER SELECT,"STENT, CORONARY, DRUG-ELUTING",BOSTON SCIENTIFIC CORPORATION,2022-12-27


In [12]:
# Narrow to a specific brand + date window
bsci_jan_thru_aug_2021 = db.query_device(
    brand_name='Boston Scientific',
    start_date='2021-01-01',
    end_date='2021-08-31'
)

print(f'Boston Scientific events in 2021 Jan-Aug: {len(bsci_jan_thru_aug_2021):,}')
bsci_jan_thru_aug_2021[['MDR_REPORT_KEY', 'BRAND_NAME', 'EVENT_TYPE', 'MANUFACTURER_G1_NAME', 'DATE_RECEIVED']]

Boston Scientific events in 2021 Jan-Aug: 2


,MDR_REPORT_KEY,BRAND_NAME,EVENT_TYPE,MANUFACTURER_G1_NAME,DATE_RECEIVED
0,11731437,BOSTON SCIENTIFIC,IN,None,2021-04-21
1,11946595,BOSTON SCIENTIFIC,M,None,2021-06-04


---
## 5. Substring search <a id="5-search"></a>

`search_by_device_names()` does **case-insensitive substring matching** across a synthesized `DEVICE_NAME_CONCAT` column (BRAND_NAME | GENERIC_NAME | MANUFACTURER_D_NAME concatenated). This is useful because MAUDE entries are often inconsistent, with names of medical devices often appearing in one of these three columns. 

| Criteria format | Logic |
|---|---|
| `'term'` | single substring |
| `['a', 'b']` | a **OR** b |
| `[['a', 'b'], 'c']` | (a **AND** b) **OR** c |
| `{'group1': ..., 'group2': ...}` | grouped (see [section 6](#6-grouped)) |

In [13]:
# Simple substring — anything with "venous stent" in any name field
venous_stents = db.search_by_device_names('venous stent')
print(f'Events matching "venous stent": {len(venous_stents):,}')
venous_stents[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'DATE_RECEIVED']]

Events matching "venous stent": 320


,MDR_REPORT_KEY,BRAND_NAME,GENERIC_NAME,DATE_RECEIVED
0,12081956,VENOVO VENOUS STENT,VENOUS STENT,2021-06-29
1,12065779,VENOVO VENOUS STENT,VENOUS STENT,2021-06-25
2,12065693,VENOVO VENOUS STENT,VENOUS STENT,2021-06-25
3,11998000,VENOVO VENOUS STENT,VENOUS STENT,2021-06-15
4,50101348,GORE® VIABAHN® FORTEGRA Venous Stent,"Stent, Superficial Femoral Artery",2026-07-31
...,...,...,...,...
315,24934139,VENOVO VENOUS STENT,VENOUS STENT,2026-04-21
316,24912731,GORE® VIABAHN® FORTEGRA VENOUS STENT,"STENT, VENA CAVA",2026-04-17
317,24912647,GORE® VIABAHN® FORTEGRA VENOUS STENT,"STENT, VENA CAVA",2026-04-17
318,24016620,VENOVO VENOUS STENT,VENOUS STENT,2026-01-08


In [14]:
# OR logic — venous stent OR iliac stent
venous_iliac = db.search_by_device_names(['venous stent', 'iliac stent'])
print(f'Venous OR iliac stents: {len(venous_iliac):,}')
venous_iliac['GENERIC_NAME'].value_counts().head(10)

Venous OR iliac stents: 701


GENERIC_NAME
SYSTEM, ENDOVASCULAR GRAFT, AORTIC ANEURYSM TR           330
VENOUS STENT                                             290
SYSTEM, ENDOVASCULAR GRAFT, AORTIC ANEURYSM TREATMENT     49
STENT, ILIAC VEIN                                         13
STENT, VENA CAVA                                           8
Stent, Superficial Femoral Artery                          4
VENOVO STENT                                               3
ILIAC STENT GRAFT EXTENDER                                 1
Venous Stent                                               1
VENOVO VENOUS STENT                                        1
Name: count, dtype: int64

In [16]:
# AND logic — must contain both "argon" AND "cleaner" (avoids false positives)
argon_cleaner = db.search_by_device_names([['argon','cleaner']])
print(f'Argon AND Cleaner events: {len(argon_cleaner):,}')
argon_cleaner[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'MANUFACTURER_D_NAME', 'DATE_RECEIVED']]

Argon AND Cleaner events: 14


,MDR_REPORT_KEY,BRAND_NAME,GENERIC_NAME,MANUFACTURER_D_NAME,DATE_RECEIVED
0,12425255,CLEANER15 ROTATIONAL THROMBECTOMY SYSTEM 7F X ...,CLEANER,ARGON MEDICAL DEVICES,2021-09-03
1,15123048,CLEANER XT ROTATIONAL THROMBECTOMY SYSTEM 6F X...,CLEANER XT,ARGON MEDICAL DEVICES,2022-07-28
2,15122938,CLEANER XT ROTATIONAL THROMBECTOMY SYSTEM 6F X...,CLEANER XT,ARGON MEDICAL DEVICES,2022-07-28
3,13523715,CLEANER XT ROTATIONAL THROMBECTOMY SYSTEM 6F X...,CLEANER XT,ARGON MEDICAL DEVICES,2022-02-14
4,11166970,CLEANER 7F X 65CM 15MM AMPLITUDE,CLEANER15,ARGON MEDICAL DEVICES,2021-01-13
5,14111566,CLEANER XT ROTATIONAL THROMBECTOMY SYSTEM 6F X...,CLEANER XT,ARGON MEDICAL DEVICES,2022-04-14
6,13251527,CLEANER XT ROTATIONAL THROMBECTOMY SYSTEM 6F X...,CLEANER XT,ARGON MEDICAL DEVICES,2022-01-13
7,12306537,CLEANER15 ROTATIONAL THROMBECTOMY SYSTEM,"CATHETER, CONTINUOUS FLUSH","ARGON MEDICAL DEVICES, INC.",2021-08-11
8,14841950,CLEANER15 ROTATIONAL THROMBECTOMY SYSTEM 7F X ...,CLEANER15,ARGON MEDICAL DEVICES,2022-06-28
9,12870775,CLEANER XT ROTATIONAL THROMBECTOMY SYSTEM 6F X...,CLEANER,ARGON MEDICAL DEVICES,2021-11-24


In [17]:
narratives = db.get_narratives(argon_cleaner['MDR_REPORT_KEY'])
narratives.get('MDR_')

In [18]:
narratives_with_date = narratives.merge(
    argon_cleaner[['MDR_REPORT_KEY', 'DATE_RECEIVED']].drop_duplicates(),
    on='MDR_REPORT_KEY', how='left'
)

with open('narratives_argon_cleaner.tmp', 'w') as f:
    for key, group in narratives_with_date.groupby('MDR_REPORT_KEY'):
        date = group['DATE_RECEIVED'].iloc[0]
        f.write(f"=== MDR_REPORT_KEY: {key} | DATE_RECEIVED: {date} ===\n")
        for text in group['FOI_TEXT'].unique():
            f.write(f"{text}\n\n")
        f.write("\n")

In [19]:
mask = narratives['FOI_TEXT'].str.contains(r'\b(vein|venous)\b', case=False, na=False)
n = narratives.loc[mask, 'MDR_REPORT_KEY'].nunique()
total = narratives['MDR_REPORT_KEY'].nunique()
print(f"{n} / {total} MDR_REPORT_KEYs have 'vein' or 'venous' in at least one FOI_TEXT")

4 / 14 MDR_REPORT_KEYs have 'vein' or 'venous' in at least one FOI_TEXT


/var/folders/0k/f52qtjsd3qq59dy3nwf1sgg40000gp/T/ipykernel_79046/3094143374.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = narratives['FOI_TEXT'].str.contains(r'\b(vein|venous)\b', case=False, na=False)


In [20]:
# Combined: (argon AND cleaner) OR (angiojet) OR (thrombectomy)
thrombectomy_broad = db.search_by_device_names(
    [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy']
)
print(f'Broad rotational thrombectomy search: {len(thrombectomy_broad):,}')
thrombectomy_broad[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME']].head(10)

Broad rotational thrombectomy search: 333


,MDR_REPORT_KEY,BRAND_NAME,GENERIC_NAME
0,15663163,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY"
1,14804199,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY"
2,14787025,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY"
3,14780364,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY"
4,14738484,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY"
5,14734797,ANGIOJET SOLENT OMNI,"CATHETER, EMBOLECTOMY"
6,14734459,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY"
7,13592791,ANGIOJET SOLENT PROXI,"CATHETER, CONTINUOUS FLUSH"
8,13584340,ANGIOJET ULTRA 5000A,"CATHETER, CORONARY, ATHERECTOMY"
9,13572872,ANGIOJET SOLENT OMNI,"CATHETER, EMBOLECTOMY"


---
## 6. Grouped search across device classes <a id="6-grouped"></a>

A **dict** of criteria runs multiple searches at once and labels each result with a `search_group` column. Useful for comparative studies.

In [21]:
grouped = db.search_by_device_names({
    'venous_stents':  ['venous stent', 'venous stenting'],
    'thrombectomy':   [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy'],
})

print(f'Total events: {len(grouped):,}')
print('\nEvents by group:')
print(grouped['search_group'].value_counts().to_string())

Total events: 651

Events by group:
search_group
thrombectomy     331
venous_stents    320


In [22]:
grouped[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'search_group']].head(15)

,MDR_REPORT_KEY,BRAND_NAME,GENERIC_NAME,search_group
0,12694967,VENOVO VENOUS STENT,VENOUS STENT,venous_stents
1,11730581,VENOVO VENOUS STENT,VENOUS STENT,venous_stents
2,11730452,VENOVO VENOUS STENT,VENOUS STENT,venous_stents
3,11730427,VENOVO VENOUS STENT,VENOUS STENT,venous_stents
4,11730309,VENOVO VENOUS STENT,VENOUS STENT,venous_stents
5,11722903,VENOVO VENOUS STENT,VENOVO STENT,venous_stents
6,11722705,VENOVO VENOUS STENT,VENOUS STENT,venous_stents
7,11722561,VENOVO VENOUS STENT,VENOUS STENT,venous_stents
8,11722501,VENOVO VENOUS STENT,VENOUS STENT,venous_stents
9,11705657,VENOVO VENOUS STENT,VENOUS STENT,venous_stents


---
## 7. Event narratives <a id="7-narratives"></a>

`get_narratives()` fetches the free-text FOI_TEXT descriptions for a set of MDR_REPORT_KEYs.

In [23]:
narratives = db.get_narratives(thrombectomy_broad['MDR_REPORT_KEY'])
print(f'Narratives retrieved: {len(narratives):,}')
narratives.head(5)

Narratives retrieved: 787


,MDR_REPORT_KEY,FOI_TEXT
0,11108890,IT WAS REPORTED THAT ERROR MESSAGES OCCURRED A...
1,11132412,IT WAS REPORTED THAT THE PROCEDURE WAS CANCELL...
2,11139889,(B)(6).
3,11139889,IT WAS REPORTED THAT THE PROCEDURE WAS CANCELL...
4,11146592,"ELDERLY MALE WITH HISTORY OF DIABETES, HYPERTE..."


In [24]:
# Read a few narratives in full
for _, row in narratives.head(3).iterrows():
    print(f"=== MDR {row['MDR_REPORT_KEY']} ===")
    print(row['FOI_TEXT'][:500])
    print()

=== MDR 11108890 ===
IT WAS REPORTED THAT ERROR MESSAGES OCCURRED AND THE PROCEDURE WAS CANCELLED. AN ANGIOJET ULTRA SYSTEM CONSOLE WAS USED FOR A THROMBECTOMY PROCEDURE. DURING PROCEDURE, IT WAS NOTED THAT THE DEVICE SHOWED AN ERROR 60 AND ERROR 6. THE PATIENT HAD BEEN SEDATED AND PROCEDURE WAS CANCELLED. NO PATIENT COMPLICATIONS WERE REPORTED AND PATIENT'S STATUS WAS STABLE.

=== MDR 11132412 ===
IT WAS REPORTED THAT THE PROCEDURE WAS CANCELLED. AN ANGIOJET CATHETER WAS USED FOR A THROMBECTOMY PROCEDURE. HOWEVER, IT WAS FOUND THAT THE CATHETER WAS LEAKING BLOOD INTO THE CONSOLE BEFORE THE PROCEDURE. THE PROCEDURE WAS CANCELLED. NO PATIENT COMPLICATIONS WERE REPORTED.

=== MDR 11139889 ===
(B)(6).



---
## 8. Enrich with patient outcomes <a id="8-patient"></a>

`enrich_with_patient_data()` left-joins the patient outcomes table. `SEQUENCE_NUMBER_OUTCOME` codes:
- `D` Death · `IN` Injury · `H` Hospitalization · `LT` Life Threatening · `OT` Other · `R` Required Intervention

In [25]:
enriched = db.enrich_with_patient_data(thrombectomy_broad)

with_outcome = enriched.dropna(subset=['SEQUENCE_NUMBER_OUTCOME'])
print(f'Events with patient outcome data: {len(with_outcome):,} / {len(enriched):,}')
with_outcome[['MDR_REPORT_KEY', 'BRAND_NAME', 'SEQUENCE_NUMBER_OUTCOME']].head(10)

Events with patient outcome data: 6 / 333


,MDR_REPORT_KEY,BRAND_NAME,SEQUENCE_NUMBER_OUTCOME
168,25243055,ANGIOJET? ZELANTEDVT?,H
290,25767157,ANGIOJET SOLENT OMNI,D
292,24075496,ANGIOJET SOLENT OMNI,R
330,25829537,ANGIOJET? ZELANTEDVT?,H; R
331,25796597,ANGIOJET? SOLENT? OMNI,R
332,24100958,CLEANER VAC SYSTEM 18F X 115CM,R


In [26]:
OUTCOME_LABELS = {
    'D': 'Death', 'IN': 'Injury', 'H': 'Hospitalization',
    'LT': 'Life Threatening', 'OT': 'Other', 'R': 'Required Intervention'
}

outcome_counts = (
    with_outcome['SEQUENCE_NUMBER_OUTCOME']
    .str.split(';')
    .explode()
    .map(lambda c: OUTCOME_LABELS.get(c, c))
    .value_counts()
)
outcome_counts

SEQUENCE_NUMBER_OUTCOME
Required Intervention    3
Hospitalization          2
Death                    1
 R                       1
Name: count, dtype: int64

---
## 9. Enrich with device problem codes <a id="9-problems"></a>

`enrich_with_problems()` left-joins the device problem code table (available from 2019 onwards).

In [27]:
with_problems = db.enrich_with_problems(thrombectomy_broad)

problem_counts = (
    with_problems
    .dropna(subset=['DEVICE_PROBLEM_CODE'])
    ['DEVICE_PROBLEM_CODE']
    .value_counts()
)
print(f'Events with a problem code: {problem_counts.sum():,}')
print('\nTop problem codes:')
problem_counts.head(10)

BinderException: Binder Error: Referenced column "MDR_REPORT_KEY" not found in FROM clause!
Candidate bindings: "column0"

LINE 1: SELECT * FROM problems WHERE MDR_REPORT_KEY IN (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ...
                                     ^

---
## 10. Trend analysis by year <a id="10-trends"></a>

`get_trends_by_year()` counts events per calendar year. When results include a `search_group` column it breaks down by group automatically.

In [28]:
# Overall year-over-year trend
trends = db.get_trends_by_year(thrombectomy_broad)
print('Rotational thrombectomy — events by year:')
print(trends.to_string(index=False))

Rotational thrombectomy — events by year:
 year  event_count
 2021          180
 2022          127
 2026           26


In [29]:
# Per-group trends
group_trends = db.get_trends_by_year(grouped)
print('Events by year and device group:')
print(group_trends.to_string(index=False))

Events by year and device group:
 search_group  year  event_count
 thrombectomy  2021          179
 thrombectomy  2022          126
 thrombectomy  2026           26
venous_stents  2021          255
venous_stents  2022           15
venous_stents  2026           50


In [30]:
# Pivot for side-by-side comparison
if 'search_group' in group_trends.columns:
    pivot = (
        group_trends
        .pivot(index='year', columns='search_group', values='count')
        .fillna(0).astype(int)
    )
    print(pivot.to_string())

KeyError: 'count'

---
## 11. Raw SQL <a id="11-sql"></a>

`db.query()` exposes DuckDB directly. Tables available: `master`, `device`, `text`, `patient`, `problems`.

In [31]:
# Event type breakdown across all loaded data
db.query("""
    SELECT
        EVENT_TYPE,
        CASE EVENT_TYPE
            WHEN 'D'  THEN 'Death'
            WHEN 'IN' THEN 'Injury'
            WHEN 'M'  THEN 'Malfunction'
            WHEN 'O'  THEN 'Other'
            ELSE EVENT_TYPE
        END AS label,
        COUNT(*) AS n
    FROM master
    GROUP BY EVENT_TYPE
    ORDER BY n DESC
""")

,EVENT_TYPE,label,n
0,M,Malfunction,4641110
1,IN,Injury,2381139
2,D,Death,27801
3,NaN,NaN,541
4,O,Other,439
5,*,*,168


In [ ]:
# Top 10 manufacturers by adverse event volume (master joined to device)
db.query("""
    SELECT
        d.MANUFACTURER_D_NAME,
        COUNT(DISTINCT m.MDR_REPORT_KEY) AS events
    FROM master m
    JOIN device d USING (MDR_REPORT_KEY)
    GROUP BY d.MANUFACTURER_D_NAME
    ORDER BY events DESC
    LIMIT 10
""")

In [ ]:
# Parameterized query — safe for user-supplied input
db.query(
    "SELECT MDR_REPORT_KEY, BRAND_NAME, DATE_RECEIVED FROM device WHERE DEVICE_REPORT_PRODUCT_CODE = ? LIMIT 10",
    params=['NIQ']
)

---
## 12. PRISMA-compliant search strategy <a id="12-prisma"></a>

`DeviceSearchStrategy` encodes a full PRISMA workflow in a single, version-controllable object:

1. **Broad search** → maximize recall
2. **Narrow search** → high-precision subset (no review needed)
3. **Difference** (`broad − narrow`) → events requiring manual adjudication
4. **Exclusion patterns** → auto-remove known off-topic hits
5. **Manual overrides** → force include/exclude specific MDR_REPORT_KEYs

Strategies serialize to YAML for reproducibility and version control.

In [32]:
strategy = DeviceSearchStrategy(
    name='rotational_thrombectomy',
    description='Rotational thrombectomy devices for DVT — MAUDE systematic review',
    version='1.0.0',
    author='Your Name',

    # Broad: catch all possible device name variations
    broad_criteria=[
        ['argon', 'cleaner'],
        'angiojet',
        'rotational thrombectomy',
    ],

    # Narrow: unambiguous hits that need no manual review
    narrow_criteria=[
        ['argon', 'cleaner'],
        'angiojet',
    ],

    known_variants=[
        'Cleaner XT', 'Cleaner 15', 'Cleaner 10',
        'AngioJet Ultra', 'AngioJet Spiroflex',
    ],

    # Auto-exclude these from the manual review set
    exclusion_patterns=['biliary', 'ultrasonic', 'coronary'],

    search_rationale=(
        'Broad adds generic thrombectomy terms to capture unlabeled variants. '
        'Narrow restricts to named brands (Argon Cleaner, AngioJet) as high confidence. '
        'Broad-only events are reviewed manually; coronary/biliary uses excluded.'
    ),
)

print(strategy)

DeviceSearchStrategy(name='rotational_thrombectomy', description='Rotational thrombectomy devices for DVT — MAUDE systematic review', version='1.0.0', author='Your Name', created_at=datetime.datetime(2026, 8, 27, 12, 31, 20, 345290), updated_at=datetime.datetime(2026, 8, 27, 12, 31, 20, 345313), broad_criteria=[['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy'], narrow_criteria=[['argon', 'cleaner'], 'angiojet'], known_variants=['Cleaner XT', 'Cleaner 15', 'Cleaner 10', 'AngioJet Ultra', 'AngioJet Spiroflex'], exclusion_patterns=['biliary', 'ultrasonic', 'coronary'], inclusion_overrides=[], exclusion_overrides=[], search_rationale='Broad adds generic thrombectomy terms to capture unlabeled variants. Narrow restricts to named brands (Argon Cleaner, AngioJet) as high confidence. Broad-only events are reviewed manually; coronary/biliary uses excluded.')


In [33]:
included, excluded, needs_review = strategy.apply(db, start_date='2022-01-01')

print(f'Included (high confidence): {len(included):,}')
print(f'Needs manual review:        {len(needs_review):,}')
print(f'Excluded:                   {len(excluded):,}')

Included (high confidence): 153
Needs manual review:        0
Excluded:                   0


In [34]:
# PRISMA flow counts
counts = strategy.get_prisma_counts(included, excluded, needs_review)

print('PRISMA Flow Diagram')
print('───────────────────')
print(f"  Identified (broad search):   {counts['broad_total']:,}")
print(f"  Sent for manual review:      {counts['needs_manual_review']:,}")
print(f"  Excluded after review:       {counts['excluded_total']:,}")
print(f"  Final included:              {counts['final_included']:,}")

PRISMA Flow Diagram
───────────────────
  Identified (broad search):   153
  Sent for manual review:      0
  Excluded after review:       0
  Final included:              153


In [35]:
included[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'DATE_RECEIVED']].head(10)

,MDR_REPORT_KEY,BRAND_NAME,GENERIC_NAME,DATE_RECEIVED
0,15246611,ANGIOJET SOLENT OMNI,"CATHETER, EMBOLECTOMY",2022-08-17
1,15245081,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY",2022-08-17
2,15224356,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY",2022-08-15
3,15217130,ANGIOJET SOLENT OMNI,"CATHETER, EMBOLECTOMY",2022-08-12
4,14566088,ANGIOJET,"CATHETER, CORONARY, ATHERECTOMY",2022-05-31
5,14550006,ANGIOJET SOLENT OMNI,"CATHETER, EMBOLECTOMY",2022-05-31
6,14521835,ANGIOJET® SPIROFLEX®,"CATHETER, CORONARY, ATHERECTOMY",2022-05-27
7,14474743,ANGIOJET SPIROFLEX,"CATHETER, EMBOLECTOMY",2022-05-23
8,14466340,ANGIOJET SOLENT DISTA,"CATHETER, EMBOLECTOMY",2022-05-21
9,14466191,ANGIOJET ULTRA SYSTEM CONSOLE,"CATHETER, CORONARY, ATHERECTOMY",2022-05-21


In [36]:
# Events needing manual adjudication
needs_review[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'MANUFACTURER_D_NAME']].head(10)

,MDR_REPORT_KEY,BRAND_NAME,GENERIC_NAME,MANUFACTURER_D_NAME


### Save strategy to YAML (version control friendly)

In [37]:
strategy.to_yaml('rotational_thrombectomy_v1.yaml')
print(strategy.to_yaml())

name: rotational_thrombectomy
description: Rotational thrombectomy devices for DVT — MAUDE systematic review
version: 1.0.0
author: Your Name
created_at: '2026-08-27T12:31:20.345290'
updated_at: '2026-08-27T12:31:20.345313'
broad_criteria:
- - argon
  - cleaner
- angiojet
- rotational thrombectomy
narrow_criteria:
- - argon
  - cleaner
- angiojet
known_variants:
- Cleaner XT
- Cleaner 15
- Cleaner 10
- AngioJet Ultra
- AngioJet Spiroflex
exclusion_patterns:
- biliary
- ultrasonic
- coronary
inclusion_overrides: []
exclusion_overrides: []
search_rationale: Broad adds generic thrombectomy terms to capture unlabeled variants.
  Narrow restricts to named brands (Argon Cleaner, AngioJet) as high confidence. Broad-only
  events are reviewed manually; coronary/biliary uses excluded.



In [ ]:
# Reload and re-apply (identical results)
reloaded = DeviceSearchStrategy.from_yaml('rotational_thrombectomy_v1.yaml')
inc2, exc2, rev2 = reloaded.apply(db, start_date='2022-01-01')
assert len(inc2) == len(included)
print('Round-trip verified ✓')

### Manual overrides — after adjudication

After reviewing `needs_review`, record decisions as overrides in a new strategy version:

In [ ]:
# Example: reviewer decided to include two borderline cases and exclude one
if len(needs_review) >= 2:
    to_include = needs_review['MDR_REPORT_KEY'].iloc[:2].tolist()
    to_exclude = needs_review['MDR_REPORT_KEY'].iloc[2:3].tolist() if len(needs_review) > 2 else []
else:
    to_include, to_exclude = [], []

strategy_v2 = DeviceSearchStrategy(
    name='rotational_thrombectomy',
    description='Rotational thrombectomy — after manual review pass 1',
    version='2.0.0',
    author='Your Name',
    broad_criteria=[['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy'],
    narrow_criteria=[['argon', 'cleaner'], 'angiojet'],
    exclusion_patterns=['biliary', 'ultrasonic', 'coronary'],
    inclusion_overrides=to_include,
    exclusion_overrides=to_exclude,
    search_rationale='v2: manual review pass 1 complete.',
)

inc_v2, exc_v2, rev_v2 = strategy_v2.apply(db, start_date='2022-01-01')
counts_v2 = strategy_v2.get_prisma_counts(inc_v2, exc_v2, rev_v2)

print('PRISMA counts after manual adjudication:')
for k, v in counts_v2.items():
    print(f'  {k}: {v:,}')

---

In [ ]:
db.close()

# 8/27: git init the above and codebase, then rm the prisma search strategy stuff from this file and the codebase, git that, publish without prisma stuff (too much! buggy!)